# SHAP Explainability — Deep Dive Notebook

This notebook provides an in-depth analysis of SHAP values:
- Global feature importance
- SHAP beeswarm / summary plots
- Waterfall plots for individual customers
- Archetype-level SHAP analysis
- Automated narrative generation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

from src.data.generate_data import generate_dataset
from src.features.feature_engineering import engineer_features, MODEL_FEATURES
from src.explainability.shap_engine import SHAPExplainer
from src.nba_engine.archetype_classifier import classify_archetypes

plt.rcParams['figure.dpi'] = 120
print('Setup complete ✓')

## 1. Train a Quick Model

In [ ]:
df_raw = generate_dataset(n=5000)
df_eng = engineer_features(df_raw)

X = df_eng[MODEL_FEATURES].fillna(0).values
y = df_eng['churned'].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(X)), test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    class_weight='balanced', random_state=42, verbose=-1
)
model.fit(X_train_s, y_train)

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
print(f'Model ROC-AUC: {auc:.4f}')

## 2. SHAP Explainer Setup

In [ ]:
explainer = SHAPExplainer(model, feature_names=MODEL_FEATURES)
explainer.fit(X_train_s[:1000])  # background dataset

# Compute SHAP values on test set
shap_values = explainer.explain(X_test_s)
print(f'SHAP values shape: {shap_values.shape}')
print(f'Base value (expected churn prob): {explainer.base_value:.4f}')

## 3. Global Feature Importance

In [ ]:
importance_df = explainer.get_global_importance()
print('Top 15 features by mean |SHAP|:')
print(importance_df.head(15).to_string(index=False))

# Bar plot
top15 = importance_df.head(15).sort_values('mean_abs_shap')
colors = ['#e94560' if v > top15['mean_abs_shap'].median() else '#457b9d'
          for v in top15['mean_abs_shap']]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15['feature'], top15['mean_abs_shap'], color=colors)
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/shap/global_importance_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Summary (Beeswarm) Plot

In [ ]:
shap.summary_plot(
    shap_values, X_test_s,
    feature_names=MODEL_FEATURES,
    max_display=15,
    show=True
)

## 5. Individual Customer Waterfall Plots

In [ ]:
churn_probs = model.predict_proba(X_test_s)[:, 1]

# Find a high-risk customer
high_risk_idx = np.where(churn_probs >= 0.70)[0]
if len(high_risk_idx) > 0:
    idx = high_risk_idx[0]
    cid = df_eng.iloc[idx_test[idx]]['customer_id']
    prob = churn_probs[idx]
    print(f'Customer: {cid} | Churn Probability: {prob:.2%}')

    exp = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.base_value,
        data=X_test_s[idx],
        feature_names=MODEL_FEATURES
    )
    shap.plots.waterfall(exp, max_display=12)
else:
    print('No high-risk customers in this test set — lower the threshold')

## 6. Automated Narrative Generation

In [ ]:
# Generate explanations for top 5 highest-risk customers
top5_idx = np.argsort(churn_probs)[::-1][:5]
customer_ids = [df_eng.iloc[idx_test[i]]['customer_id'] for i in top5_idx]

for rank, (i, cid) in enumerate(zip(top5_idx, customer_ids)):
    explanation = explainer.explain_customer(
        customer_id=cid,
        idx=i,
        X_row=X_test_s[i],
        churn_prob=churn_probs[i],
        top_n=4
    )
    print(f'\n--- Rank {rank+1} ---')
    print(explanation['narrative'])
    print(f'  → Primary Driver: {explanation["primary_driver"]}')

## 7. SHAP Analysis by Archetype

In [ ]:
# Add SHAP values to test dataframe
df_test = df_eng.iloc[idx_test].copy().reset_index(drop=True)
df_test['churn_probability'] = churn_probs
df_test_hr = df_test[df_test['churn_probability'] >= 0.5].copy()
df_classified = classify_archetypes(df_test_hr)

print('Average churn probability by archetype:')
print(df_classified.groupby('churn_archetype')['churn_probability'].agg(['mean','count']).round(3))

print('\nTop feature score by archetype (avg):')
score_cols = ['engagement_decay_score','friction_score','pricing_sensitivity_score','growth_pressure_score']
print(df_classified.groupby('churn_archetype')[score_cols].mean().round(3))

## 8. SHAP Interaction Effects

In [ ]:
# Dependence plot: engagement decay score vs churn
eng_idx = MODEL_FEATURES.index('engagement_decay_score')
billing_idx = MODEL_FEATURES.index('billing_page_visits')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    X_test_s[:, eng_idx], shap_values[:, eng_idx],
    c=churn_probs, cmap='RdYlGn_r', alpha=0.4, s=10
)
axes[0].set_xlabel('engagement_decay_score (scaled)')
axes[0].set_ylabel('SHAP value')
axes[0].set_title('SHAP Dependence: Engagement Decay')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

axes[1].scatter(
    X_test_s[:, billing_idx], shap_values[:, billing_idx],
    c=churn_probs, cmap='RdYlGn_r', alpha=0.4, s=10
)
axes[1].set_xlabel('billing_page_visits (scaled)')
axes[1].set_ylabel('SHAP value')
axes[1].set_title('SHAP Dependence: Billing Page Visits')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('../reports/shap/shap_dependence_plots.png', dpi=150, bbox_inches='tight')
plt.show()